In [1]:

from pathlib import Path

import numpy as np
import pandas as pd

root = Path.cwd()

if root.name == "notebooks":
    root = root.parent

# Load asset returns
asset_returns = pd.read_csv(
    root / "data" / "processed" / "asset_returns.csv",
    parse_dates=["Date"]
)

# Load portfolio returns
portfolio_returns = pd.read_csv(
    root / "data" / "processed" / "portfolio_returns.csv",
    parse_dates=["Date"]
)

# Load rolling forecasts
models = {}

files = {
    "Historical": "historical_var_es_forecasts.csv",
    "Parametric": "parametric_var_es_forecasts.csv",
    "Monte Carlo": "monte_carlo_var_es_forecasts.csv"
}

for name, filename in files.items():
    models[name] = pd.read_csv(
        root / "results" / filename,
        parse_dates=["Date"]
    )

print("Asset returns:", asset_returns.shape)
print("Portfolio returns:", portfolio_returns.shape)

for name, df in models.items():
    print(name, df.shape)


Asset returns: (4232, 6)
Portfolio returns: (4232, 6)
Historical (3732, 9)
Parametric (3732, 11)
Monte Carlo (3732, 8)


In [2]:

# Final Audit: Dates and Realized Returns

tickers = ["SPY", "QQQ", "IWM", "EFA", "EEM"]
weights = np.array([0.40, 0.25, 0.15, 0.10, 0.10])

window = 500

# Recalculate portfolio returns independently
recalculated = (
    asset_returns[tickers].to_numpy() @ weights
)

# Expected forecast observations start after 500 returns
expected_dates = (
    asset_returns["Date"].iloc[window:].reset_index(drop=True)
)

expected_returns = recalculated[window:]

assert len(expected_dates) == 3732

for name, df in models.items():

    assert df["Date"].reset_index(drop=True).equals(
        expected_dates
    )

    assert np.allclose(
        df["actual_return"].to_numpy(),
        expected_returns,
        atol=1e-10,
        rtol=0
    )

    print(f"{name}: Dates and returns validated!")

print("\nAll forecast alignment checks passed!")


Historical: Dates and returns validated!
Parametric: Dates and returns validated!
Monte Carlo: Dates and returns validated!

All forecast alignment checks passed!


In [3]:

# Final Audit: Independent Parametric Forecast Recalculation

from scipy.stats import norm

parametric_df = models["Parametric"]

check_positions = [500, 1500, 3000, 4231]

audit_results = []

for t in check_positions:

    # Historical data strictly before forecast date
    historical_window = asset_returns[tickers].iloc[
        t-window:t
    ]

    mu = historical_window.mean().to_numpy()
    cov = historical_window.cov().to_numpy()

    mu_p = weights @ mu
    sigma_p = np.sqrt(weights @ cov @ weights)

    recalculated_var_99 = (
        -mu_p + norm.ppf(0.99) * sigma_p
    )

    forecast_row = parametric_df.iloc[t-window]

    stored_var_99 = forecast_row["var_99"]

    difference = (
        recalculated_var_99 - stored_var_99
    )

    audit_results.append({
        "Date": forecast_row["Date"],
        "Recalculated VaR 99": recalculated_var_99,
        "Stored VaR 99": stored_var_99,
        "Difference": difference
    })

audit_df = pd.DataFrame(audit_results)

display(audit_df)

assert np.allclose(
    audit_df["Recalculated VaR 99"],
    audit_df["Stored VaR 99"],
    atol=1e-10,
    rtol=0
)

print("Independent Parametric Forecast Audit Passed!")


,Date,Recalculated VaR 99,Stored VaR 99,Difference
0,2011-12-28,0.033106,0.033106,3.469447e-17
1,2015-12-18,0.020283,0.020283,6.938894e-17
2,2021-12-03,0.036447,0.036447,6.938894e-17
3,2026-10-30,0.025143,0.025143,6.591949e-17


Independent Parametric Forecast Audit Passed!


In [4]:

# Step 2A: Independent Historical VaR / ES Audit

def independent_historical_var_es(returns, confidence):

    r = np.sort(np.asarray(returns, dtype=float))

    alpha = 1 - confidence

    # Historical VaR with linear interpolation
    var = -np.quantile(
        r,
        alpha,
        method="linear"
    )

    # Fractional-tail ES
    tail_size = len(r) * alpha
    full_count = int(np.floor(tail_size))
    fractional_weight = tail_size - full_count

    tail_sum = -r[:full_count].sum()

    if fractional_weight > 1e-12:
        tail_sum += (
            fractional_weight * (-r[full_count])
        )

    es = tail_sum / tail_size

    return var, es


historical_df = models["Historical"]

check_positions = [500, 1500, 3000, 4231]

hs_audit_results = []

for t in check_positions:

    # Use only 500 observations before day t
    historical_window = recalculated[t-window:t]

    forecast_row = historical_df.iloc[t-window]

    for confidence in [0.95, 0.99]:

        level = int(round(confidence * 100))

        var, es = independent_historical_var_es(
            historical_window,
            confidence
        )

        hs_audit_results.append({
            "Date": forecast_row["Date"],
            "Confidence": f"{level}%",
            "Recalculated VaR": var,
            "Stored VaR": forecast_row[f"var_{level}"],
            "Recalculated ES": es,
            "Stored ES": forecast_row[f"es_{level}"]
        })

hs_audit_df = pd.DataFrame(hs_audit_results)

hs_audit_df["VaR Difference"] = (
    hs_audit_df["Recalculated VaR"]
    - hs_audit_df["Stored VaR"]
)

hs_audit_df["ES Difference"] = (
    hs_audit_df["Recalculated ES"]
    - hs_audit_df["Stored ES"]
)

display(hs_audit_df)

assert np.allclose(
    hs_audit_df["Recalculated VaR"],
    hs_audit_df["Stored VaR"],
    atol=1e-10,
    rtol=0
)

assert np.allclose(
    hs_audit_df["Recalculated ES"],
    hs_audit_df["Stored ES"],
    atol=1e-10,
    rtol=0
)

print("Independent Historical VaR / ES Audit Passed!")


,Date,Confidence,Recalculated VaR,Stored VaR,Recalculated ES,Stored ES,VaR Difference,ES Difference
0,2011-12-28,95%,0.024551,0.024551,0.034679,0.034679,-3.291117e-14,3.878842e-15
1,2011-12-28,99%,0.040437,0.040437,0.051578,0.051578,-9.860168e-15,-7.945034e-15
2,2015-12-18,95%,0.014592,0.014592,0.020749,0.020749,3.884566e-14,6.127043e-15
3,2015-12-18,99%,0.023390,0.023390,0.030250,0.030250,5.474787e-14,2.206915e-14
4,2021-12-03,95%,0.023208,0.023208,0.041231,0.041231,3.885781e-15,1.106060e-14
5,2021-12-03,99%,0.049724,0.049724,0.080854,0.080854,1.933176e-14,1.003364e-14
6,2026-10-30,95%,0.016362,0.016362,0.024193,0.024193,-1.220204e-14,-9.200973e-15
7,2026-10-30,99%,0.029541,0.029541,0.042399,0.042399,-2.996214e-14,-2.692985e-14


Independent Historical VaR / ES Audit Passed!


In [9]:
import sys
from pathlib import Path

# Find project root
root = Path.cwd()

if root.name == "notebooks":
    root = root.parent

# Add project root to Python import path
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print("Project root:", root)

from src.historical_var import historical_var, historical_es

print("Import successful!")

Project root: /Users/tzuling/Desktop/HW/self project/portfolio-var-es-backtesting
Import successful!


In [10]:
# import src
# Step 3A: Monte Carlo Reproducibility Audit

from src.historical_var import historical_var, historical_es

mc_df = models["Monte Carlo"]

n_simulations = 100_000
base_seed = 42

# Recreate the exact daily random streams
seed_sequence = np.random.SeedSequence(base_seed)
child_seeds = seed_sequence.spawn(len(asset_returns) - window)

# Check selected forecast positions
check_positions = [500, 1500, 3000, 4231]

mc_audit_results = []

for t in check_positions:

    # Historical data strictly before forecast date
    historical_window = asset_returns[tickers].iloc[
        t-window:t
    ]

    mu = historical_window.mean().to_numpy()
    cov = historical_window.cov().to_numpy()

    # Portfolio normal parameters
    mu_p = weights @ mu
    sigma_p = np.sqrt(weights @ cov @ weights)

    # Reproduce the original random stream for this day
    i = t - window
    rng = np.random.default_rng(child_seeds[i])

    simulated_portfolio = rng.normal(
        loc=mu_p,
        scale=sigma_p,
        size=n_simulations
    )

    forecast_row = mc_df.iloc[i]

    # Recalculate VaR and ES
    for confidence in [0.95, 0.99]:

        level = int(round(confidence * 100))

        var = historical_var(
            simulated_portfolio, confidence
        )

        es = historical_es(
            simulated_portfolio, confidence
        )

        mc_audit_results.append({
            "Date": forecast_row["Date"],
            "Confidence": f"{level}%",
            "Recalculated VaR": var,
            "Stored VaR": forecast_row[f"var_{level}"],
            "Recalculated ES": es,
            "Stored ES": forecast_row[f"es_{level}"]
        })

mc_audit_df = pd.DataFrame(mc_audit_results)

mc_audit_df["VaR Difference"] = (
    mc_audit_df["Recalculated VaR"]
    - mc_audit_df["Stored VaR"]
)

mc_audit_df["ES Difference"] = (
    mc_audit_df["Recalculated ES"]
    - mc_audit_df["Stored ES"]
)

display(mc_audit_df)


,Date,Confidence,Recalculated VaR,Stored VaR,Recalculated ES,Stored ES,VaR Difference,ES Difference
0,2011-12-28,95%,0.023423,0.023423,0.029483,0.029483,2.775558e-17,0.000000e+00
1,2011-12-28,99%,0.033317,0.033317,0.038116,0.038116,1.387779e-17,7.632783e-17
2,2015-12-18,95%,0.014279,0.014279,0.017914,0.017914,4.857226e-17,6.938894e-18
3,2015-12-18,99%,0.020311,0.020311,0.023117,0.023117,3.469447e-17,6.938894e-18
4,2021-12-03,95%,0.025459,0.025459,0.032112,0.032112,3.122502e-17,6.245005e-17
5,2021-12-03,99%,0.036375,0.036375,0.041807,0.041807,0.000000e+00,1.387779e-17
6,2026-10-30,95%,0.017614,0.017614,0.022241,0.022241,3.469447e-18,1.387779e-17
7,2026-10-30,99%,0.025259,0.025259,0.028975,0.028975,7.632783e-17,4.163336e-17


In [11]:
assert np.allclose(
    mc_audit_df["Recalculated VaR"],
    mc_audit_df["Stored VaR"],
    atol=1e-10,
    rtol=0
)

assert np.allclose(
    mc_audit_df["Recalculated ES"],
    mc_audit_df["Stored ES"],
    atol=1e-10,
    rtol=0
)

print("Monte Carlo Reproducibility Audit Passed!")

Monte Carlo Reproducibility Audit Passed!


In [12]:

# Step 4A: Full-Sample Parametric VaR / ES Audit

from scipy.stats import norm

parametric_df = models["Parametric"]

# Parameters already saved by the Parametric model
mu = parametric_df["portfolio_mean"].to_numpy()
sigma = parametric_df["portfolio_volatility"].to_numpy()

assert np.isfinite(mu).all()
assert np.isfinite(sigma).all()
assert (sigma > 0).all()

for confidence in [0.95, 0.99]:

    level = int(round(confidence * 100))
    z = norm.ppf(confidence)

    # Recalculate analytical Gaussian VaR
    calculated_var = -mu + z * sigma

    # Recalculate analytical Gaussian ES
    calculated_es = (
        -mu
        + sigma * norm.pdf(z) / (1 - confidence)
    )

    # Compare to saved forecasts
    var_diff = np.abs(
        calculated_var - parametric_df[f"var_{level}"]
    )

    es_diff = np.abs(
        calculated_es - parametric_df[f"es_{level}"]
    )

    print(f"\n{level}% Parametric Audit")
    print(f"Maximum VaR difference: {var_diff.max():.2e}")
    print(f"Maximum ES difference:  {es_diff.max():.2e}")

    assert np.allclose(
        calculated_var,
        parametric_df[f"var_{level}"],
        atol=1e-10,
        rtol=0
    )

    assert np.allclose(
        calculated_es,
        parametric_df[f"es_{level}"],
        atol=1e-10,
        rtol=0
    )

print("\nFull-Sample Parametric Audit Passed!")



95% Parametric Audit
Maximum VaR difference: 1.91e-16
Maximum ES difference:  2.26e-16

99% Parametric Audit
Maximum VaR difference: 2.43e-16
Maximum ES difference:  2.57e-16

Full-Sample Parametric Audit Passed!


In [13]:

# Step 4B: Mathematical Consistency Audit

audit_summary = []

for name, df in models.items():

    assert df["Date"].is_unique
    assert df["Date"].is_monotonic_increasing

    var95 = df["var_95"].to_numpy()
    var99 = df["var_99"].to_numpy()

    es95 = df["es_95"].to_numpy()
    es99 = df["es_99"].to_numpy()

    assert np.isfinite(
        np.column_stack([var95, var99, es95, es99])
    ).all()

    # Risk measure consistency
    assert (var99 >= var95).all()
    assert (es95 >= var95).all()
    assert (es99 >= var99).all()
    assert (es99 >= es95).all()

    audit_summary.append({
        "Model": name,
        "Forecasts": len(df),
        "Date Alignment": "Passed",
        "Risk Ordering": "Passed",
        "Missing Values": int(df.isna().sum().sum())
    })

audit_summary_df = pd.DataFrame(audit_summary)

display(audit_summary_df)

assert (audit_summary_df["Missing Values"] == 0).all()

print("Full-Sample Mathematical Consistency Audit Passed!")


,Model,Forecasts,Date Alignment,Risk Ordering,Missing Values
0,Historical,3732,Passed,Passed,0
1,Parametric,3732,Passed,Passed,0
2,Monte Carlo,3732,Passed,Passed,0


Full-Sample Mathematical Consistency Audit Passed!
